### Example of working with data extracted by the Sniffer.

The Sniffer will output a local SQLite database. This is a file in the `/data` folder with the name `<project_name>.db`. Here we demonstrate how to work with that local database. 

Note: I would recommend copying the database file to another location on your machine and starting a new repository to work with the data extract.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
#import matplotlib.pyplot as plt
#import os
#os.chdir('../streamlit/')
#from utilities import LocalDatabaseWrapper
#os.chdir('../development')

# %matplotlib_inline

In [2]:
project_name = "smartt_bmj_open_plus"
database_path = Path("../data")

You can connect directly to the database using the SQLite3 library. You can then, for example, use Pandas to read the results of a SQL query to a dataframe. 

In [15]:
con = sqlite3.connect(database_path / f"{project_name}.db")

In [16]:
pd.read_sql_query("select * from sqlite_master where type = 'table';", con)

,type,name,tbl_name,rootpage,sql
0,table,schema,schema,2,"CREATE TABLE ""schema"" (\n""mapping_complete"" IN..."
1,table,info,info,5,"CREATE TABLE ""info"" (\n""name"" TEXT,\n ""databa..."
2,table,clinical_unit_ids,clinical_unit_ids,6,"CREATE TABLE ""clinical_unit_ids"" (\n""Select"" I..."
3,table,fact_table_init_status,fact_table_init_status,7,"CREATE TABLE ""fact_table_init_status"" (\n""tabl..."
4,table,PtAssessmentInterventions,PtAssessmentInterventions,9,"CREATE TABLE ""PtAssessmentInterventions"" (\n""i..."
5,table,PtLabResultInterventions,PtLabResultInterventions,79,"CREATE TABLE ""PtLabResultInterventions"" (\n""in..."
6,table,PtDemographicInterventions,PtDemographicInterventions,109,"CREATE TABLE ""PtDemographicInterventions"" (\n""..."
7,table,table_definitions,table_definitions,116,"CREATE TABLE ""table_definitions"" (\n""tableType..."
8,table,distinct_attributes,distinct_attributes,119,"CREATE TABLE ""distinct_attributes"" (\n""attribu..."
9,table,example_attribute_data,example_attribute_data,120,"CREATE TABLE ""example_attribute_data"" (\n""attr..."


#### You will find the database contains a bunch of tables that are used by the Sniffer. You will mainly want to work with:

* schema: a copy of your project schema
* final_mapping: the contains the mapping that you created in the Sniffer between your schema variables and the ICCA definitions
* full_extract: the large data table containing the full extract for all patients and all variables

In [17]:
con.close()

The Sniffer also has a wrapper class for working with the local database. It handles the connection and has some helper methods. It can be used like so:

In [23]:
import sys
sys.path.append('../application/')
from utilities import LocalDatabaseWrapper

In [24]:
db = LocalDatabaseWrapper(
    database_path / f"{project_name}.db"
)

In [26]:
# This would run the same table query as above to list all the tables:
_ = db.query_pd("select * from sqlite_master where type = 'table';")

#### It is recommended to create whatever index you need to speed up your queries, especially for larger extracts. For example:

In [ ]:
# Commented out once run because it takes some time
# db.insert_query(
#     "CREATE INDEX intervention_attribute ON full_extract (interventionId, attributeId);"
# )

In [28]:
# Check that index has been added (if not, uncomment and run above cell or create your own index).
db.query_pd("PRAGMA index_list('full_extract');")

,seq,name,unique,origin,partial
0,0,encounter,0,c,0
1,1,intervention_attribute,0,c,0


So this is the mapping that was produced by the Sniffer:

In [30]:
final_mapping = db.query_pd(
    "select * from final_mapping;"
    
)
final_mapping .head()

,schemaVariable,interventionId,interventionLongLabel,priority,attributeId,attributeShortLabel,tableName
0,Gender,2631,Gender,0,11050,Gender,PtDemographic
1,BMI,2645,Body Mass Index,0,3723,BMI,PtDemographic
2,Weight,2110,Weight (admission),0,3588,Admit weight,PtDemographic
3,Weight,2110,Weight (admission),0,40554,Admit weight,PtDemographic
4,Weight,4615,Weight (Admission),0,1636,Weight,PtDemographic


#### We can check, for example, the mapping for heart rate:

In [31]:
final_mapping[final_mapping.schemaVariable =='Heart rate']

,schemaVariable,interventionId,interventionLongLabel,priority,attributeId,attributeShortLabel,tableName
60,Heart rate,3361,Heart Rate,0,12754,HR,PtAssessment
61,Heart rate,7838,Heart Rate,0,27308,HR,PtAssessment
62,Heart rate,7838,Heart Rate,0,27309,HR,PtAssessment
63,Heart rate,7838,Heart Rate,0,27312,HR,PtAssessment


The index that was added should speed up running queries for any interventionId and attributeId pair on the `full_extract` table:

In [32]:
df = db.query_pd(
    "select * from full_extract where interventionId = 7838 and attributeId = 27312;"
)

In [33]:
df.head()

,attributeId,interventionId,encounterId,attributeShortLabel,attributeLongLabel,clinicalUnitId,terseForm,verboseForm,valueNumber,valueString,...,chartTime,storeTime,utcChartTime,careProviderId,tableTypeId,bedId,lowerNormal,upperNormal,attributeConceptLabel,attributeConceptCode
0,27312,7838,1583,HR,Heart Rate.HR.Heart Rate,5,75,75 bpm,75.0,None,...,2015-06-16 16:00:00,2015-06-16 16:00:34.287000,2015-06-16 15:00:00,4,4,35,50.0,120.0,Heart rate (observable entity),364075005
1,27312,7838,1583,HR,Heart Rate.HR.Heart Rate,5,74,74 bpm,74.0,None,...,2015-06-16 17:00:00,2015-06-16 17:00:11.627000,2015-06-16 16:00:00,4,4,35,50.0,120.0,Heart rate (observable entity),364075005
2,27312,7838,1583,HR,Heart Rate.HR.Heart Rate,5,76,76 bpm,76.0,None,...,2015-06-16 18:00:00,2015-06-16 18:00:15.640000,2015-06-16 17:00:00,4,4,35,50.0,120.0,Heart rate (observable entity),364075005
3,27312,7838,1583,HR,Heart Rate.HR.Heart Rate,5,75,75 bpm,75.0,None,...,2015-06-16 19:00:00,2015-06-16 19:00:13.877000,2015-06-16 18:00:00,4,4,35,50.0,120.0,Heart rate (observable entity),364075005
4,27312,7838,1583,HR,Heart Rate.HR.Heart Rate,5,78,78 bpm,78.0,None,...,2015-06-16 20:00:00,2015-06-16 20:00:15.323000,2015-06-16 19:00:00,4,4,35,50.0,120.0,Heart rate (observable entity),364075005


You can see - most of the information has been retained from the ICCA fact table. How you process this for your 